# 데이터 전처리
### Pascal VOC (XML) -> YOLO 포맷 (.txt) 변환 + Train / Val / Test 분할

In [1]:
import os
import shutil
import random
import xml.etree.ElementTree as ET
from pathlib import Path

print('[OK] 임포트 완료')

[OK] 임포트 완료


### 경로 및 설정

In [2]:
DATA_ROOT = Path(r"C:\workspace_python\deeplearning\project\pcb-defects")          # ← 원본 데이터 경로
IMAGE_DIR  = DATA_ROOT / "images"
ANNOT_DIR  = DATA_ROOT / "Annotations"
OUTPUT_DIR = Path("./pcb_yolo_dataset")     # ← YOLO 학습용 데이터 저장 경로

CLASSES = [
    "missing_hole",
    "mouse_bite",
    "open_circuit",
    "short",
    "spur",
    "spurious_copper"
]
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}

SPLIT_RATIO = {"train": 0.7, "val": 0.2, "test": 0.1}
SEED = 42

print(f'원본 데이터: {DATA_ROOT}')
print(f'출력 경로:   {OUTPUT_DIR}')
print(f'클래스 목록: {CLASSES}')

원본 데이터: C:\workspace_python\deeplearning\project\pcb-defects
출력 경로:   pcb_yolo_dataset
클래스 목록: ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']


### VOC XML -> YOLO txt 변환 함수

In [3]:
def voc_to_yolo(xml_path):
    """Pascal VOC XML → YOLO 정규화 좌표 변환"""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    img_w = int(root.find("size/width").text)
    img_h = int(root.find("size/height").text)

    lines = []
    for obj in root.findall("object"):
        cls_name = obj.find("name").text.strip().lower().replace(" ", "_")
        if cls_name not in CLASS2IDX:
            print(f"[WARN] 알 수 없는 클래스: '{cls_name}' → 스킵")
            continue

        cls_idx = CLASS2IDX[cls_name]
        xmin = int(obj.find("bndbox/xmin").text)
        ymin = int(obj.find("bndbox/ymin").text)
        xmax = int(obj.find("bndbox/xmax").text)
        ymax = int(obj.find("bndbox/ymax").text)

        # 경계 클리핑 (이미지 범위 초과 방지)
        xmin = max(0, min(xmin, img_w))
        xmax = max(0, min(xmax, img_w))
        ymin = max(0, min(ymin, img_h))
        ymax = max(0, min(ymax, img_h))

        # YOLO 정규화 변환
        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        width    = (xmax - xmin) / img_w
        height   = (ymax - ymin) / img_h

        lines.append(f"{cls_idx} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    return lines

print('[OK] 변환 함수 정의 완료')

[OK] 변환 함수 정의 완료


### 디렉토리 구조 생성

In [4]:
for split in ["train", "val", "test"]:
    (OUTPUT_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

print(f'[OK] 디렉토리 생성 완료:')
for split in ["train", "val", "test"]:
    print(f'     {OUTPUT_DIR}/{split}/images  &  labels')

[OK] 디렉토리 생성 완료:
     pcb_yolo_dataset/train/images  &  labels
     pcb_yolo_dataset/val/images  &  labels
     pcb_yolo_dataset/test/images  &  labels


### 이미지 - XML 쌍 수집

In [6]:
import os
from pathlib import Path

# 1. 최상위 루트 경로 설정 (이미지와 XML이 모두 포함된 가장 상위 폴더)
DATA_ROOT = Path(r"C:\workspace_python\deeplearning\project\pcb-defects")

# 2. 모든 이미지 파일의 경로를 미리 딕셔너리에 저장 (이름: 경로)
# rglob을 사용하여 하위 폴더를 모두 뒤집니다.
all_image_paths = {f.stem: f for f in DATA_ROOT.rglob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]}
print(f"[INFO] 발견된 총 이미지 수: {len(all_image_paths)}개")

# 3. 모든 XML 파일을 찾아서 이미지와 매칭
xml_files = list(DATA_ROOT.rglob("*.xml"))
pairs = []

for xf in xml_files:
    # xml 파일명(xf.stem)이 이미지 딕셔너리에 있는지 확인
    if xf.stem in all_image_paths:
        pairs.append((all_image_paths[xf.stem], xf))
    else:
        # 매칭 안 된 경우 로그 출력 (디버깅용)
        # print(f"[WARN] 이미지 없음: {xf.name}")
        pass

print(f"[INFO] 유효한 이미지-XML 쌍: {len(pairs)}개")

# 샘플 확인 (제대로 매칭되었는지 경로 출력)
if pairs:
    img_p, xml_p = pairs[0]
    print(f"\n[SAMPLE MATCH]\nImage: {img_p}\nXML  : {xml_p}")

[INFO] 발견된 총 이미지 수: 703개
[INFO] 유효한 이미지-XML 쌍: 693개

[SAMPLE MATCH]
Image: C:\workspace_python\deeplearning\project\pcb-defects\PCB_DATASET\rotation\Missing_hole_rotation\01_missing_hole_01.jpg
XML  : C:\workspace_python\deeplearning\project\pcb-defects\PCB_DATASET\Annotations\Missing_hole\01_missing_hole_01.xml


### 데이터 분할 및 변환 + 복사

In [7]:
random.seed(SEED)
random.shuffle(pairs)

n       = len(pairs)
n_train = int(n * SPLIT_RATIO["train"])
n_val   = int(n * SPLIT_RATIO["val"])

splits = {
    "train": pairs[:n_train],
    "val"  : pairs[n_train:n_train + n_val],
    "test" : pairs[n_train + n_val:]
}

stats = {s: {"ok": 0, "skip": 0} for s in splits}

for split, pair_list in splits.items():
    for img_path, xml_path in pair_list:
        yolo_lines = voc_to_yolo(xml_path)
        if not yolo_lines:          # 객체 없는 파일 스킵
            stats[split]["skip"] += 1
            continue

        # 이미지 복사
        shutil.copy2(img_path, OUTPUT_DIR / split / "images" / img_path.name)

        # YOLO 라벨 저장
        dst_lbl = OUTPUT_DIR / split / "labels" / (img_path.stem + ".txt")
        with open(dst_lbl, "w") as f:
            f.write("\n".join(yolo_lines))

        stats[split]["ok"] += 1

print(f"\n[RESULT] 데이터 분할 완료")
print(f"{'Split':<8} {'OK':>6} {'Skip':>6}")
print("-" * 25)
for split, s in stats.items():
    print(f"{split:<8} {s['ok']:>6} {s['skip']:>6}")


[RESULT] 데이터 분할 완료
Split        OK   Skip
-------------------------
train       485      0
val         138      0
test         70      0


### dataset.yaml 생성

In [8]:
yaml_content = f"""# PCB Defect Detection - YOLO Dataset Config
path: {OUTPUT_DIR.resolve()}

train: train/images
val:   val/images
test:  test/images

nc: {len(CLASSES)}
names:
"""
for cls in CLASSES:
    yaml_content += f"  - {cls}\n"

yaml_path = OUTPUT_DIR / "dataset.yaml"
with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"[SAVE] dataset.yaml → {yaml_path}")
print()
print(yaml_content)

[SAVE] dataset.yaml → pcb_yolo_dataset\dataset.yaml

# PCB Defect Detection - YOLO Dataset Config
path: C:\workspace_python\deeplearning\project\pcb_yolo_dataset

train: train/images
val:   val/images
test:  test/images

nc: 6
names:
  - missing_hole
  - mouse_bite
  - open_circuit
  - short
  - spur
  - spurious_copper



### 변환 결과 검증 (샘플 확인)

In [9]:
print('[VERIFY] YOLO 라벨 샘플 확인 (train 3개):')
label_files = list((OUTPUT_DIR / "train" / "labels").glob("*.txt"))
for lf in label_files[:3]:
    print(f"\n  {lf.name}:")
    with open(lf) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_idx = int(parts[0])
            print(f"    class={CLASSES[cls_idx]}({cls_idx})  "
                  f"cx={float(parts[1]):.4f}  cy={float(parts[2]):.4f}  "
                  f"w={float(parts[3]):.4f}  h={float(parts[4]):.4f}")

print("\n[DONE] 전처리 완료! dataset.yaml을 학습에 사용하세요.")

[VERIFY] YOLO 라벨 샘플 확인 (train 3개):

  01_missing_hole_02.txt:
    class=missing_hole(0)  cx=0.8626  cy=0.1671  w=0.0218  h=0.0416
    class=missing_hole(0)  cx=0.7864  cy=0.5243  w=0.0132  h=0.0359
    class=missing_hole(0)  cx=0.5051  cy=0.5227  w=0.0175  h=0.0340

  01_missing_hole_03.txt:
    class=missing_hole(0)  cx=0.8818  cy=0.3209  w=0.0162  h=0.0277
    class=missing_hole(0)  cx=0.4507  cy=0.4871  w=0.0168  h=0.0296
    class=missing_hole(0)  cx=0.6950  cy=0.3789  w=0.0175  h=0.0240

  01_missing_hole_05.txt:
    class=missing_hole(0)  cx=0.2742  cy=0.2090  w=0.0185  h=0.0422
    class=missing_hole(0)  cx=0.1042  cy=0.2106  w=0.0244  h=0.0504
    class=missing_hole(0)  cx=0.3589  cy=0.6803  w=0.0138  h=0.0353
    class=missing_hole(0)  cx=0.5422  cy=0.2686  w=0.0211  h=0.0429

[DONE] 전처리 완료! dataset.yaml을 학습에 사용하세요.
